# Tests for the BHNR STL reward (`stl.py`)

Run top to bottom (Kernel → Restart & Run All). Every check prints `PASS` / `FAIL` right under the cell that ran it, and the last cell summarizes them by section. Nothing is written to CSV/JSON.

**Put this notebook in the same folder as `stl.py` and `race_env.py`.** The tests build their own small synthetic course, so they run anywhere. The real-course sections (E2 and F) run only if `purt_course.yaml` is found.

| section | what it tests | needs a course? |
|---|---|---|
| **A** | the pure math: `until_robustness`, window schedule, `BHNRWindow`, `h_label` | no |
| **B** | your 8 distance scenarios (K closer / further / constant × K+1 closer / further / constant) | no |
| **C** | the "good, good, bad" example from the design doc (how long a bad step stays masked) | no |
| **D** | `STLOrderMonitor` on a small course: schedule, reward, `H_t`, stage changes, delta mode, metric | synthetic |
| **E** | hover-aware weight scaling (`robustness_weight`, `hover_gain_per_step`), the `2.58` constant (`calibrate_hover_rho`), and why delta mode cannot be farmed | E2: real |
| **F** | smoke test: short Q-learning in both modes on the real course | real |
| **G** | `sweep.py`: experiment lists and job ids, `train`/`run_job` determinism, resume, worker processes, analysis helpers | synthetic |

## Setup

In [3]:
import json
import os
import sys
import tempfile
import time
from collections import defaultdict

import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())          # run from the folder that contains stl.py and race_env.py
import stl
import sweep
from stl import (BHNRWindow, H_NEG, H_NONE, H_POS, STL_REWARDS, STLOrderMonitor, bhnr_series, calibrate_hover_rho, h_label,
                 hover_gain_per_step, n_h_classes, robustness_weight, until_robustness)
from race_env import RaceEnv

print("stl.py   loaded from:", os.path.abspath(stl.__file__))
print("sweep.py loaded from:", os.path.abspath(sweep.__file__))
print("stl public API      :", ", ".join(stl.__all__))

# ---- tiny test harness: every check prints PASS/FAIL and is counted in the summary at the end --------------------
CHECKS, _section = [], ["-"]
def section(name):
    _section[0] = name
    print(f"\n=== {name} ===")
def check(name, condition, detail=""):
    CHECKS.append((_section[0], name, bool(condition)))
    print(f"  {'PASS' if condition else 'FAIL'}  {name}" + (f"   [{detail}]" if detail else ""))
    return bool(condition)

pd.set_option("display.width", 200)
NAMES = {H_NONE: "NONE", H_NEG: "NEG", H_POS: "POS"}

stl.py   loaded from: /Users/rzhang/Projects/RL_planner/RL Tutorial/Demo Results/stl.py
sweep.py loaded from: /Users/rzhang/Projects/RL_planner/RL Tutorial/Demo Results/sweep.py
stl public API      : H_NONE, H_NEG, H_POS, REWARD_MODES, STL_REWARDS, until_robustness, h_label, n_h_classes, BHNRWindow, bhnr_series, HOVER_RHO_HAT, hover_gain_per_step, robustness_weight, calibrate_hover_rho, STLOrderMonitor


### Synthetic course
Four vertical gates, 1 m grid, no pylon in the way of the test path. The agent starts on the middle of G1 at (5, 10); flying **RIGHT** crosses G1 on step 1 (so the stage-1 clause is live: target **G2** at x = 17, avoid **G3** at x = 28) and crosses G2 on step 12.

In [4]:
COURSE_YAML = """
gate_margin_m: 1.0
bounds_rect: {min_x: 0, max_x: 40, min_y: 0, max_y: 30}
pylons:
  - {x: 5,  y: 8,  r: 0.5, name: P0}
  - {x: 5,  y: 12, r: 0.5, name: P1}
  - {x: 17, y: 8,  r: 0.5, name: P2}
  - {x: 17, y: 12, r: 0.5, name: P3}
  - {x: 28, y: 16, r: 0.5, name: P4}
  - {x: 28, y: 20, r: 0.5, name: P5}
  - {x: 17, y: 24, r: 0.5, name: P6}
  - {x: 17, y: 28, r: 0.5, name: P7}
gates:
  - {p1: 0, p2: 1, name: G1}
  - {p1: 2, p2: 3, name: G2}
  - {p1: 4, p2: 5, name: G3}
  - {p1: 6, p2: 7, name: G4}
"""
SYN_COURSE = os.path.join(tempfile.mkdtemp(), "synthetic_course.yaml")
with open(SYN_COURSE, "w") as f:
    f.write(COURSE_YAML)
RIGHT, UP, LEFT, DOWN = 0, 1, 2, 3

def make_monitor(course=SYN_COURSE, weight=1.0, r1=3.0, r2=3.0, noise=0, **kw):
    env = RaceEnv(course, CELL_SIZE_M=1.0, MAX_STEPS=400, NOISE=noise)
    return STLOrderMonitor(env, REWARDS={**STL_REWARDS, "stl_order": weight}, r1=r1, r2=r2, **kw)

def fly(monitor, actions):
    """reset, then take the actions; returns one (obs, reward, done, truncated, info) per step"""
    monitor.reset()
    return [monitor.step(a) for a in actions]

m = make_monitor()
print("grid cells:", m.env.GRID_NX, "x", m.env.GRID_NY, "| gates:", [g["name"] for g in m.env.GATES])
print("start obs :", m.reset()[0], "(grid index, q, H_t)")

grid cells: 41 x 31 | gates: ['G1', 'G2', 'G3', 'G4']
start obs : (415, 0, 0) (grid index, q, H_t)


## A. Pure math (`until_robustness`, windows, labels)
No environment involved: these are plain functions and a small class.

In [5]:
section("A1. until_robustness equals the definition sup_t' min(rho_target(t'), inf_{t''<=t'} rho_avoid(t''))")
rng = np.random.default_rng(0)
worst = 0.0
for _ in range(500):
    n = int(rng.integers(1, 9))
    a, b = rng.normal(size=n) * 5, rng.normal(size=n) * 5
    brute = max(min(a[t], min(b[: t + 1])) for t in range(n))            # direct double loop
    worst = max(worst, abs(until_robustness(a, b) - brute))
check("500 random sequences match a brute-force double loop", worst < 1e-12, f"max abs diff = {worst:.1e}")
check("single state: min(target, avoid)", until_robustness([2.0], [5.0]) == 2.0 and until_robustness([5.0], [2.0]) == 2.0)
check("avoid violated early caps everything after it", until_robustness([-1, 9, 9], [-4, 9, 9]) == -4.0,
      "rho_avoid = -4 at t=0 -> running inf is -4 for every later state, so even rho_target = 9 yields -4")
check("...but a late violation only affects later states", until_robustness([5, 1, 1], [9, 9, -4]) == 5.0,
      "state 0 already reached the target (5) before the avoid gate was entered")


=== A1. until_robustness equals the definition sup_t' min(rho_target(t'), inf_{t''<=t'} rho_avoid(t'')) ===
  PASS  500 random sequences match a brute-force double loop   [max abs diff = 0.0e+00]
  PASS  single state: min(target, avoid)
  PASS  avoid violated early caps everything after it   [rho_avoid = -4 at t=0 -> running inf is -4 for every later state, so even rho_target = 9 yields -4]
  PASS  ...but a late violation only affects later states   [state 0 already reached the target (5) before the avoid gate was entered]


True

In [6]:
section("A2. bad inputs raise ValueError")
for label, fn in [("unequal lengths", lambda: until_robustness([1, 2], [1])),
                  ("empty", lambda: until_robustness([], [])),
                  ("window_len = 0", lambda: BHNRWindow(0, 0)),
                  ("overlap == window_len", lambda: BHNRWindow(3, 3)),
                  ("negative overlap", lambda: BHNRWindow(3, -1))]:
    try:
        fn(); ok, msg = False, "no error"
    except ValueError as e:
        ok, msg = True, str(e)[:70]
    check(f"{label} -> ValueError", ok, msg)


=== A2. bad inputs raise ValueError ===
  PASS  unequal lengths -> ValueError   [rho_target and rho_avoid must be equal-length, non-empty 1-D sequences]
  PASS  empty -> ValueError   [rho_target and rho_avoid must be equal-length, non-empty 1-D sequences]
  PASS  window_len = 0 -> ValueError   [window_len must be >= 1, got 0]
  PASS  overlap == window_len -> ValueError   [overlap must satisfy 0 <= overlap < window_len, got overlap=3, window_]
  PASS  negative overlap -> ValueError   [overlap must satisfy 0 <= overlap < window_len, got overlap=-1, window]


In [7]:
section("A3. window schedule: which states does each window cover, and when does it close?")
def span_table(configs, n_states=13):
    rows = []
    for L, ov in configs:
        stride = L - ov
        # rho_target = -index and rho_avoid huge  =>  window robustness = -(index of the window's FIRST state)
        out = bhnr_series(-np.arange(n_states, dtype=float), np.full(n_states, 1e9), L, ov)
        ends = [t for t, _ in out]
        starts = [int(-r) for _, r in out]
        expected_ends = list(range(L, n_states, stride))
        rows.append(dict(window_len=L, overlap=ov, stride=stride,
                         windows=" ".join(f"[{s}..{e}]" for s, e in zip(starts, ends)),
                         ok=(ends == expected_ends and all(e - s == L for s, e in zip(starts, ends)))))
    return pd.DataFrame(rows)

sched = span_table([(3, 0), (3, 1), (3, 2), (1, 0), (4, 1), (6, 5)])
print(sched.to_string(index=False))
check("every window spans window_len actions and closes every `stride` steps", sched["ok"].all())


=== A3. window schedule: which states does each window cover, and when does it close? ===
 window_len  overlap  stride                                                                                  windows   ok
          3        0       3                                                             [0..3] [3..6] [6..9] [9..12] True
          3        1       2                                                      [0..3] [2..5] [4..7] [6..9] [8..11] True
          3        2       1                 [0..3] [1..4] [2..5] [3..6] [4..7] [5..8] [6..9] [7..10] [8..11] [9..12] True
          1        0       1 [0..1] [1..2] [2..3] [3..4] [4..5] [5..6] [6..7] [7..8] [8..9] [9..10] [10..11] [11..12] True
          4        1       3                                                                    [0..4] [3..7] [6..10] True
          6        5       1                                      [0..6] [1..7] [2..8] [3..9] [4..10] [5..11] [6..12] True
  PASS  every window spans window_len actions an

True

In [8]:
section("A4. BHNRWindow lifecycle (start / push / stop)")
w = BHNRWindow(window_len=3, overlap=1)
check("inactive window ignores pushes", w.push(1.0, 1.0) is None and not w.active)
w.start(rho_target=-5.0, rho_avoid=9.0)
trace = []
for t, rt in enumerate([-4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0], start=1):    # approach: rho_target rises by 1 per step
    trace.append((t, rt, w.push(rt, 9.0)))
print(pd.DataFrame(trace, columns=["state t", "rho_target", "window closes with rho"]).to_string(index=False))
closes = [t for t, _, r in trace if r is not None]
check("closes at t = 3, 5, 7", closes == [3, 5, 7], f"closes at {closes}")
check("each closing returns the window's best (last) state", [r for _, _, r in trace if r is not None] == [-2.0, 0.0, 2.0])
w.start(-5.0, 9.0)
check("start() restarts the clock", w.n == 0 and w.push(-4.0, 9.0) is None)
w.stop()
check("stop() deactivates", (not w.active) and w.push(1.0, 1.0) is None)


=== A4. BHNRWindow lifecycle (start / push / stop) ===
  PASS  inactive window ignores pushes
 state t  rho_target  window closes with rho
       1        -4.0                     NaN
       2        -3.0                     NaN
       3        -2.0                    -2.0
       4        -1.0                     NaN
       5         0.0                     0.0
       6         1.0                     NaN
       7         2.0                     2.0
  PASS  closes at t = 3, 5, 7   [closes at [3, 5, 7]]
  PASS  each closing returns the window's best (last) state
  PASS  start() restarts the clock
  PASS  stop() deactivates


True

In [9]:
section("A5. H_t labels (class 0 is reserved for 'no window closed')")
rhos = [-5.0, -0.1, 0.0, 0.1, 5.0]
tbl = pd.DataFrame({"rho(window)": rhos,
                    "default bins (0.0,)": [NAMES[h_label(r)] for r in rhos],
                    "bins [-3, 0] (class id)": [h_label(r, [-3.0, 0.0]) for r in rhos]})
print(tbl.to_string(index=False))
check("(H_NONE, H_NEG, H_POS) == (0, 1, 2)", (H_NONE, H_NEG, H_POS) == (0, 1, 2))
check("rho = 0 is NEG (boundary is not satisfied)", h_label(0.0) == H_NEG)
check("n_h_classes: 3 by default, 4 with two bins", n_h_classes() == 3 and n_h_classes([-3.0, 0.0]) == 4)
check("a real window never gets class 0", all(h_label(r, [-3.0, 0.0]) >= 1 for r in rhos))


=== A5. H_t labels (class 0 is reserved for 'no window closed') ===
 rho(window) default bins (0.0,)  bins [-3, 0] (class id)
        -5.0                 NEG                        1
        -0.1                 NEG                        2
         0.0                 NEG                        2
         0.1                 POS                        3
         5.0                 POS                        3
  PASS  (H_NONE, H_NEG, H_POS) == (0, 1, 2)
  PASS  rho = 0 is NEG (boundary is not satisfied)
  PASS  n_h_classes: 3 by default, 4 with two bins
  PASS  a real window never gets class 0


True

## B. The 8 scenarios (one 4-state window, r1 = r2 = 3)
`d_k` / `d_k1` are the distances to gate k and gate k+1 at each of the 4 states. ρ is positive when a state inside r1 of gate k is reached without first entering r2 of gate k+1.

In [10]:
section("B. scenario signs")
R1 = R2 = 3.0
SCENARIOS = {  # name: (distances to gate k, distances to gate k+1, expected sign of rho(window))
    "K closer, K+1 further":                   ([8, 6, 4, 2],   [10, 12, 14, 16], +1),
    "K closer, K+1 closer (stays > r2)":       ([8, 6, 4, 2],   [16, 14, 12, 10], +1),
    "K closer, K+1 closer (enters r2 first)":  ([8, 6, 4, 2],   [8, 6, 4, 2],     -1),
    "K further, K+1 further":                  ([8, 10, 12, 14], [10, 12, 14, 16], -1),
    "K further, K+1 closer":                   ([4, 6, 8, 10],  [14, 10, 6, 2],   -1),
    "K constant, K+1 further":                 ([6, 6, 6, 6],   [8, 10, 12, 14],  -1),
    "K constant, K+1 closer":                  ([6, 6, 6, 6],   [14, 12, 10, 8],  -1),
    "K further, K+1 constant":                 ([4, 5, 6, 7],   [10, 10, 10, 10], -1),
    "K closer, K+1 constant":                  ([8, 6, 4, 2],   [10, 10, 10, 10], +1),
}
rows = []
for name, (dk, dk1, sign) in SCENARIOS.items():
    rho_t, rho_a = R1 - np.array(dk, float), np.array(dk1, float) - R2
    rho = until_robustness(rho_t, rho_a)
    rows.append(dict(scenario=name, d_k=dk, d_k1=dk1, rho=rho, label=NAMES[h_label(rho)], expected="+" if sign > 0 else "-",
                     ok=bool(np.sign(rho) == sign)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
for _, r in res.iterrows():
    check(r["scenario"], r["ok"], f"rho = {r['rho']:+.1f}")


=== B. scenario signs ===
                              scenario             d_k             d_k1  rho label expected   ok
                 K closer, K+1 further    [8, 6, 4, 2] [10, 12, 14, 16]  1.0   POS        + True
     K closer, K+1 closer (stays > r2)    [8, 6, 4, 2] [16, 14, 12, 10]  1.0   POS        + True
K closer, K+1 closer (enters r2 first)    [8, 6, 4, 2]     [8, 6, 4, 2] -1.0   NEG        - True
                K further, K+1 further [8, 10, 12, 14] [10, 12, 14, 16] -5.0   NEG        - True
                 K further, K+1 closer   [4, 6, 8, 10]   [14, 10, 6, 2] -1.0   NEG        - True
               K constant, K+1 further    [6, 6, 6, 6]  [8, 10, 12, 14] -3.0   NEG        - True
                K constant, K+1 closer    [6, 6, 6, 6]  [14, 12, 10, 8] -3.0   NEG        - True
               K further, K+1 constant    [4, 5, 6, 7] [10, 10, 10, 10] -1.0   NEG        - True
                K closer, K+1 constant    [8, 6, 4, 2] [10, 10, 10, 10]  1.0   POS        + True
  P

## C. The design-doc example: how long does a bad step stay masked?
Two good actions (distance to gate k 9 → 6 → 3), then the agent turns away for good (12, 15, 18, ...). Because each window reports its *best* state, a window that still contains the good state keeps reporting it. Overlap shortens how long that lasts, but does not remove it.

In [11]:
section("C. masking of a bad step")
d_k = [9, 6, 3, 12, 15, 18, 21, 24, 27]
rho_t, rho_a = R1 - np.array(d_k, float), np.full(len(d_k), 50.0 - R2)
print("rho_target per state:", rho_t.tolist(), "\n")
series = {f"window_len=3, overlap={ov}": dict(bhnr_series(rho_t, rho_a, 3, ov)) for ov in (0, 1, 2)}
series["window_len=1, overlap=0"] = dict(bhnr_series(rho_t, rho_a, 1, 0))
for name, s in series.items():
    print(f"{name:<26} closes at state -> rho:", {t: round(r, 1) for t, r in s.items()})
non, ov1 = series["window_len=3, overlap=0"], series["window_len=3, overlap=1"]
check("no overlap: window [3..6] excludes the good state -> -9", non[3] == 0 and non[6] == -9)
check("overlap 1: window [2..5] still contains it -> masked (0)", ov1[3] == 0 and ov1[5] == 0)
check("overlap 1: first window past the good state is [4..7] -> -12", ov1[7] == -12)
check("shorter stride reports the drop sooner (overlap 2 shows -9 by state 6)", series["window_len=3, overlap=2"][6] == -9)


=== C. masking of a bad step ===
rho_target per state: [-6.0, -3.0, 0.0, -9.0, -12.0, -15.0, -18.0, -21.0, -24.0] 

window_len=3, overlap=0    closes at state -> rho: {3: 0.0, 6: -9.0}
window_len=3, overlap=1    closes at state -> rho: {3: 0.0, 5: 0.0, 7: -12.0}
window_len=3, overlap=2    closes at state -> rho: {3: 0.0, 4: 0.0, 5: 0.0, 6: -9.0, 7: -12.0, 8: -15.0}
window_len=1, overlap=0    closes at state -> rho: {1: -3.0, 2: 0.0, 3: 0.0, 4: -9.0, 5: -12.0, 6: -15.0, 7: -18.0, 8: -21.0}
  PASS  no overlap: window [3..6] excludes the good state -> -9
  PASS  overlap 1: window [2..5] still contains it -> masked (0)
  PASS  overlap 1: first window past the good state is [4..7] -> -12
  PASS  shorter stride reports the drop sooner (overlap 2 shows -9 by state 6)


True

## D. `STLOrderMonitor` on the synthetic course
A real `RaceEnv` wrapped by the monitor. NOISE = 0 so every rollout is deterministic.

In [12]:
section("D1. step-by-step trace, window_len=3 overlap=1, flying RIGHT (w = 2)")
def trace(mode, weight=2.0, steps=16, **kw):
    mon = make_monitor(weight=weight, window_len=3, overlap=1, reward_mode=mode, **kw)
    rows = []
    for i, (obs, r, done, trunc, info) in enumerate(fly(mon, [RIGHT] * steps), start=1):
        rows.append(dict(step=i, x=float(mon.env.grid_to_xy(obs[0])[0]), q=obs[1], H_t=NAMES[obs[2]],
                         window_rho=info["window_rho"], stl_reward=info["reward_parts"]["stl_order"],
                         reward=r, closed=info["window_closed"]))
    return pd.DataFrame(rows)

rob, dlt = trace("robustness"), trace("delta")
both = rob[["step", "x", "q", "H_t", "window_rho", "stl_reward"]].rename(columns={"stl_reward": "stl (robustness)"})
both["stl (delta)"] = dlt["stl_reward"]
print(both.to_string(index=False, float_format=lambda v: f"{v:+.1f}"))
closed = rob.loc[rob["closed"], "step"].tolist()
check("windows close on global steps 4, 6, 8, 10, 12, then 15 after the stage change", closed == [4, 6, 8, 10, 12, 15], f"closed at {closed}")
row4 = rob[rob["step"] == 4].iloc[0]
check("first window (x = 6..9, d(G2) = 11..8) has rho = 3 - 8 = -5", row4["window_rho"] == -5.0 and row4["H_t"] == "NEG")
check("robustness pays w * rho = 2 * -5 = -10 on that step", row4["stl_reward"] == -10.0)
check("H_t is NONE and the STL reward is 0 on every non-closing step",
      bool(((rob.loc[~rob["closed"], "H_t"] == "NONE") & (rob.loc[~rob["closed"], "stl_reward"] == 0.0)).all()))
check("the total reward on a closing step = neutral (-1) + STL term", rob.loc[rob["step"] == 4, "reward"].iloc[0] == -1 + -10.0)


=== D1. step-by-step trace, window_len=3 overlap=1, flying RIGHT (w = 2) ===
 step     x  q  H_t  window_rho  stl (robustness)  stl (delta)
    1  +6.0  1 NONE         NaN              +0.0         +0.0
    2  +7.0  1 NONE         NaN              +0.0         +0.0
    3  +8.0  1 NONE         NaN              +0.0         +0.0
    4  +9.0  1  NEG        -5.0             -10.0         +0.0
    5 +10.0  1 NONE         NaN              +0.0         +0.0
    6 +11.0  1  NEG        -3.0              -6.0         +4.0
    7 +12.0  1 NONE         NaN              +0.0         +0.0
    8 +13.0  1  NEG        -1.0              -2.0         +4.0
    9 +14.0  1 NONE         NaN              +0.0         +0.0
   10 +15.0  1  POS        +1.0              +2.0         +4.0
   11 +16.0  1 NONE         NaN              +0.0         +0.0
   12 +17.0  2  POS        +3.0              +6.0         +4.0
   13 +18.0  2 NONE         NaN              +0.0         +0.0
   14 +19.0  2 NONE         NaN         

True

In [13]:
section("D2. crossing a gate: the closing on the crossing step is scored, and a new window is anchored")
out = fly(make_monitor(weight=1.0, window_len=3, overlap=1), [RIGHT] * 18)
obs12, info12 = out[11][0], out[11][4]
print(f"step 12: x = {make_monitor().env.grid_to_xy(obs12[0])[0]:.0f}, q = {obs12[1]}, H_t = {NAMES[obs12[2]]}, window_rho = {info12['window_rho']:+.1f}")
check("step 12 crosses G2 (q becomes 2)", obs12[1] == 2)
check("it was also a scheduled closing: scored against the OLD stage (d(G2)=0 -> rho = +3, POS)",
      info12["window_closed"] and info12["window_rho"] == 3.0 and info12["h_label"] == H_POS)
closed = [i for i, o in enumerate(out, 1) if o[4]["window_closed"]]
check("new window is anchored at the crossing state: next closings at 15 and 17", closed[-2:] == [15, 17], f"closed at {closed}")


=== D2. crossing a gate: the closing on the crossing step is scored, and a new window is anchored ===
step 12: x = 17, q = 2, H_t = POS, window_rho = +3.0
  PASS  step 12 crosses G2 (q becomes 2)
  PASS  it was also a scheduled closing: scored against the OLD stage (d(G2)=0 -> rho = +3, POS)
  PASS  new window is anchored at the crossing state: next closings at 15 and 17   [closed at [4, 6, 8, 10, 12, 15, 17]]


True

In [14]:
section("D3. no live clause -> no windows, no STL reward")
mon = make_monitor(first_stage=1)
mon.reset()
check("stage 0 is skipped by default (targets() == (None, None))", mon.targets() == (None, None))
out = fly(make_monitor(first_stage=3), [RIGHT] * 20)       # n_gates - 1 = 3: no stage has a clause
check("first_stage=3: nothing ever closes", all(not o[4]["window_closed"] for o in out))
check("first_stage=3: STL reward is always 0 and H_t always NONE",
      all(o[4]["reward_parts"]["stl_order"] == 0.0 and o[0][2] == H_NONE for o in out))


=== D3. no live clause -> no windows, no STL reward ===
  PASS  stage 0 is skipped by default (targets() == (None, None))
  PASS  first_stage=3: nothing ever closes
  PASS  first_stage=3: STL reward is always 0 and H_t always NONE


True

In [15]:
section("D4. delta mode pays the change between consecutive windows")
out = fly(make_monitor(weight=1.0, window_len=3, overlap=1, reward_mode="delta"), [RIGHT] * 11)
pay = [(i, o[4]["window_rho"], o[4]["reward_parts"]["stl_order"]) for i, o in enumerate(out, 1) if o[4]["window_closed"]]
print(pd.DataFrame(pay, columns=["step", "window_rho", "delta-mode reward"]).to_string(index=False, float_format=lambda v: f"{v:+.1f}"))
check("first window of a stage pays 0 (nothing to compare with)", pay[0][2] == 0.0)
check("then +2 per window (the agent gains 2 m per stride)", [p[2] for p in pay[1:]] == [2.0, 2.0, 2.0])


=== D4. delta mode pays the change between consecutive windows ===
 step  window_rho  delta-mode reward
    4        -5.0               +0.0
    6        -3.0               +2.0
    8        -1.0               +2.0
   10        +1.0               +2.0
  PASS  first window of a stage pays 0 (nothing to compare with)
  PASS  then +2 per window (the agent gains 2 m per stride)


True

In [16]:
section("D5. the full-trajectory metric rho_order does not depend on the window settings")
rows = []
for L, ov in [(1, 0), (2, 0), (3, 1), (5, 2)]:
    out = fly(make_monitor(window_len=L, overlap=ov), [RIGHT] * 11)
    rows.append(dict(window_len=L, overlap=ov, H=out[-1][4]["H"], rho_order=out[-1][4]["rho_order"], rho_order_prev=out[-2][4]["rho_order"]))
met = pd.DataFrame(rows)
print(met.to_string(index=False))
print("(stage 1, x = 7..16: d(G2) = 10..1, so the sup is 3 - 1 = 2 at the last step and 1 the step before)")
check("H = rho_order = 2.0 for every window config", bool((met["H"] == 2.0).all() and (met["rho_order"] == 2.0).all()))
check("one step earlier it was 1.0", bool((met["rho_order_prev"] == 1.0).all()))


=== D5. the full-trajectory metric rho_order does not depend on the window settings ===
 window_len  overlap   H  rho_order  rho_order_prev
          1        0 2.0        2.0             1.0
          2        0 2.0        2.0             1.0
          3        1 2.0        2.0             1.0
          5        2 2.0        2.0             1.0
(stage 1, x = 7..16: d(G2) = 10..1, so the sup is 3 - 1 = 2 at the last step and 1 the step before)
  PASS  H = rho_order = 2.0 for every window config
  PASS  one step earlier it was 1.0


True

In [17]:
section("D6. observation / state shape, hidden label, reset")
mon = make_monitor(H_BINS=[-3.0, 0.0])
obs, _ = mon.reset()
check("initial obs = (grid index, 0, H_NONE) and is hashable (Q-table key)", obs == (mon.env.state, 0, H_NONE) and isinstance(hash(obs), int))
check("observation_space[2] has 4 classes for two H_BINS", mon.observation_space[2].n == 4)

shown = fly(make_monitor(weight=2.0, window_len=3, overlap=1), [RIGHT] * 12)
hidden = fly(make_monitor(weight=2.0, window_len=3, overlap=1, observe_label=False), [RIGHT] * 12)
check("observe_label=True: the label is visible on closing steps", any(o[0][2] != H_NONE for o in shown))
check("observe_label=False: H_t never shown", all(o[0][2] == H_NONE for o in hidden))
check("observe_label=False: rewards identical", [o[1] for o in shown] == [o[1] for o in hidden])
check("observe_label=False: position and q identical", [o[0][:2] for o in shown] == [o[0][:2] for o in hidden])

mon = make_monitor(weight=2.0, window_len=3, overlap=1)
first = [(o[0], o[1]) for o in fly(mon, [RIGHT] * 12)]
second = [(o[0], o[1]) for o in fly(mon, [RIGHT] * 12)]      # fly() calls reset() again on the same monitor
check("reset() fully clears window state: the same rollout repeats exactly", first == second)


=== D6. observation / state shape, hidden label, reset ===
  PASS  initial obs = (grid index, 0, H_NONE) and is hashable (Q-table key)
  PASS  observation_space[2] has 4 classes for two H_BINS
  PASS  observe_label=True: the label is visible on closing steps
  PASS  observe_label=False: H_t never shown
  PASS  observe_label=False: rewards identical
  PASS  observe_label=False: position and q identical
  PASS  reset() fully clears window state: the same rollout repeats exactly


True

In [18]:
section("D7. invalid constructor arguments")
for label, kw in [("reward_mode='nope'", dict(reward_mode="nope")), ("overlap == window_len", dict(window_len=3, overlap=3)),
                  ("r1 = 0", dict(r1=0.0)), ("r2 < 0", dict(r2=-1.0))]:
    try:
        make_monitor(**kw); ok, msg = False, "no error"
    except ValueError as e:
        ok, msg = True, str(e)[:60]
    check(f"{label} -> ValueError", ok, msg)


=== D7. invalid constructor arguments ===
  PASS  reward_mode='nope' -> ValueError   [reward_mode must be one of ('robustness', 'delta'), got 'nop]
  PASS  overlap == window_len -> ValueError   [overlap must satisfy 0 <= overlap < window_len, got overlap=]
  PASS  r1 = 0 -> ValueError   [r1 and r2 must be positive, got r1=0.0, r2=3.0]
  PASS  r2 < 0 -> ValueError   [r1 and r2 must be positive, got r1=3.0, r2=-1.0]


## E. Weight scaling (robustness mode)
In robustness mode an agent can farm reward by oscillating next to the target gate without crossing it. If the best such hover earns `rho_hat` per window, the gain per step is `w * rho_hat / stride`. `robustness_weight(f, ...)` returns the weight that makes that gain exactly `f` per step, so hovering nets `f - 1` against the −1 neutral step cost (`f = 1` is break-even).

In [19]:
section("E1. robustness_weight and hover_gain_per_step are consistent")
configs = [(1, 0), (2, 1), (3, 1), (6, 3), (6, 5), (3, 0)]
fs = (0.5, 1, 2, 4)
rows = []
for L, ov in configs:
    rows.append({"(len, overlap)": f"({L},{ov})", "stride": L - ov,
                 **{f"f={f}": round(robustness_weight(f, L, ov), 4) for f in fs}})
print("weight w for each hover margin f:")
print(pd.DataFrame(rows).to_string(index=False))

ok = all(abs(hover_gain_per_step(robustness_weight(f, L, ov), L, ov, "robustness") - f) < 1e-9 for L, ov in configs for f in fs)
check("hover gain at robustness_weight(f) equals f for every config", ok)
check("delta mode has no hover gain at any weight", all(hover_gain_per_step(w, 3, 1, "delta") == 0.0 for w in (1, 3, 30, 1000)))
check("weight scales linearly with stride (stride 2 = 2 x stride 1)", abs(robustness_weight(1, 3, 1) - 2 * robustness_weight(1, 2, 1)) < 1e-12)


=== E1. robustness_weight and hover_gain_per_step are consistent ===
weight w for each hover margin f:
(len, overlap)  stride  f=0.5    f=1    f=2    f=4
         (1,0)       1 0.1938 0.3876 0.7752 1.5504
         (2,1)       1 0.1938 0.3876 0.7752 1.5504
         (3,1)       2 0.3876 0.7752 1.5504 3.1008
         (6,3)       3 0.5814 1.1628 2.3256 4.6512
         (6,5)       1 0.1938 0.3876 0.7752 1.5504
         (3,0)       3 0.5814 1.1628 2.3256 4.6512
  PASS  hover gain at robustness_weight(f) equals f for every config
  PASS  delta mode has no hover gain at any weight
  PASS  weight scales linearly with stride (stride 2 = 2 x stride 1)


True

In [20]:
section("E2. the 2.58 constant on the real course (skipped if purt_course.yaml is not found)")
CANDIDATES = ["purt_course.yaml", "../purt_course.yaml", "../../ros2_ws/src/pylon_2026_world/purt_course.yaml"]
REAL_COURSE = next((p for p in CANDIDATES if os.path.exists(p)), None)
print("real course:", REAL_COURSE or "NOT FOUND -> sections E2 and F are skipped")

if REAL_COURSE:
    t0 = time.perf_counter()
    rho_hat, (xy_a, xy_b) = calibrate_hover_rho(REAL_COURSE)
    print(f"best non-crossing hover: cells {np.round(xy_a, 1)} <-> {np.round(xy_b, 1)}, rho_hat = {rho_hat:.3f} per window ({time.perf_counter() - t0:.1f}s)")
    for L, ov in [(3, 1), (6, 5)]:      # any window of >= 2 states contains both oscillation cells
        print(f"  window ({L},{ov}): rho_hat = {calibrate_hover_rho(REAL_COURSE, L, ov)[0]:.3f}")
    check("measured rho_hat matches stl.HOVER_RHO_HAT = 2.58", abs(rho_hat - stl.HOVER_RHO_HAT) < 0.01, f"{rho_hat:.3f} vs {stl.HOVER_RHO_HAT}")
else:
    print("skipped")


=== E2. the 2.58 constant on the real course (skipped if purt_course.yaml is not found) ===
real course: ../../ros2_ws/src/pylon_2026_world/purt_course.yaml
best non-crossing hover: cells [-6.  1.] <-> [-6.  2.], rho_hat = 2.585 per window (0.4s)
  window (3,1): rho_hat = 2.585
  window (6,5): rho_hat = 2.585
  PASS  measured rho_hat matches stl.HOVER_RHO_HAT = 2.58   [2.585 vs 2.58]


In [21]:
section("E3. delta mode cannot be farmed by oscillating: its gain telescopes to ~0")
def oscillation_gains(L, ov, amp, n=120):
    # STL reward per step at w = 1 for an agent that oscillates next to the gate: the distance is a triangle wave
    # between 0.5 and 0.5 + amp. Returns (robustness-mode gain, delta-mode gain), measured in steady state.
    d = 0.5 + np.array([abs(((t + amp) % (2 * amp)) - amp) for t in range(n)], float)
    series = [r for _, r in bhnr_series(3.0 - d, np.full(n, 9.0), L, ov)]
    tail = np.array(series[len(series) // 2:])
    stride = L - ov
    return tail.mean() / stride, (tail[-1] - tail[0]) / ((len(tail) - 1) * stride)

rows = []
for L, ov in [(1, 0), (2, 1), (3, 1), (6, 5)]:
    for amp in (1, 2, 3, 4):
        rob_gain, delta_gain = oscillation_gains(L, ov, amp)
        rows.append({"(len, overlap)": f"({L},{ov})", "amplitude (cells)": amp, "robustness gain/step": rob_gain, "delta gain/step": delta_gain})
osc = pd.DataFrame(rows)
print(osc.to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
check("delta-mode gain stays below 0.05 per step for every oscillation", bool((osc["delta gain/step"].abs() < 0.05).all()),
      f"max = {osc['delta gain/step'].abs().max():.3f}")
check("robustness mode earns more than 0.8 per step from the same oscillations", bool((osc["robustness gain/step"] > 0.8).all()),
      f"min = {osc['robustness gain/step'].min():.2f}")


=== E3. delta mode cannot be farmed by oscillating: its gain telescopes to ~0 ===
(len, overlap)  amplitude (cells)  robustness gain/step  delta gain/step
         (1,0)                  1                +2.500           +0.000
         (1,0)                  2                +2.000           -0.017
         (1,0)                  3                +1.500           -0.017
         (1,0)                  4                +0.950           +0.034
         (2,1)                  1                +2.500           +0.000
         (2,1)                  2                +2.246           -0.017
         (2,1)                  3                +1.822           -0.017
         (2,1)                  4                +1.331           +0.034
         (3,1)                  1                +1.250           +0.000
         (3,1)                  2                +1.250           +0.000
         (3,1)                  3                +1.083           -0.017
         (3,1)                  4        

True

## F. Smoke test on the real course
150 episodes of epsilon-greedy Q-learning in each mode (same loop as the sweep). This only checks that nothing crashes and that windows, labels and rewards are actually produced during learning. It is **not** a tuning run.

In [22]:
section("F. short Q-learning on the real course, both modes")
def smoke(mode, w, L, ov, episodes=150, seed=0):
    np.random.seed(seed)
    mon = make_monitor(course=REAL_COURSE, weight=w, r1=3.0, r2=9.5, noise=0.25, first_stage=1,
                       window_len=L, overlap=ov, reward_mode=mode, H_BINS=[0.0])
    Q = defaultdict(lambda: np.random.random(4))
    closed = pos = neg = steps = 0
    stl_total, rho_finite, t0 = 0.0, 0, time.perf_counter()
    for _ in range(episodes):
        s, _ = mon.reset()
        while True:
            a = np.random.randint(4) if np.random.random() < 0.3 else int(np.argmax(Q[s]))
            s2, r, done, trunc, info = mon.step(a)
            Q[s][a] += 0.1 * (r + 0.99 * np.max(Q[s2]) * (1 - done) - Q[s][a])
            s, steps = s2, steps + 1
            stl_total += info["reward_parts"]["stl_order"]
            if info["window_closed"]:
                closed += 1; pos += info["h_label"] == H_POS; neg += info["h_label"] == H_NEG
            if done or trunc:
                break
        rho_finite += np.isfinite(info["rho_order"])
    return dict(mode=mode, w=w, window=f"({L},{ov})", episodes=episodes, steps=steps, windows_closed=closed,
                POS=int(pos), NEG=int(neg), stl_reward_total=round(stl_total, 1), eps_with_finite_rho_order=int(rho_finite),
                seconds=round(time.perf_counter() - t0, 1))

if REAL_COURSE:
    res = pd.DataFrame([smoke("robustness", round(robustness_weight(2, 1, 0), 4), 1, 0), smoke("delta", 3, 2, 1)])
    print(res.to_string(index=False))
    check("windows were closed in both modes", bool((res["windows_closed"] > 0).all()))
    check("both POS and NEG labels occurred", bool(((res["POS"] > 0) & (res["NEG"] > 0)).all()))
    check("STL reward was actually paid", bool((res["stl_reward_total"] != 0).all()))
    check("rho_order was finite in some episodes (stages were scored)", bool((res["eps_with_finite_rho_order"] > 0).all()))
else:
    print("skipped: real course not found")


=== F. short Q-learning on the real course, both modes ===
      mode      w window  episodes  steps  windows_closed  POS   NEG  stl_reward_total  eps_with_finite_rho_order  seconds
robustness 0.7752  (1,0)       150  28896           24353 2183 22170         -110724.9                        149      1.8
     delta 3.0000  (2,1)       150  31322           14501 3741 10760            5942.8                        148      1.8
  PASS  windows were closed in both modes
  PASS  both POS and NEG labels occurred
  PASS  STL reward was actually paid
  PASS  rho_order was finite in some episodes (stages were scored)


## G. `sweep.py` plumbing
The experiment runner around `stl.py`: job lists and ids, `train` / `run_job` determinism, resume, worker processes and the analysis helpers. Tiny runs (20-40 episodes) on the synthetic course, so this takes a few seconds and needs no real course.

In [23]:
section("G1. experiment lists and job ids")
sizes = {name: len(make()) for name, make in sweep.EXPERIMENTS.items()}
print("jobs per phase:", sizes)
check("phase sizes are 43 / 32 / 90 / 60 / 32", list(sizes.values()) == [43, 32, 90, 60, 32])
check("job ids are unique within each phase", all(len({sweep.job_id(j) for j in make()}) == len(make()) for make in sweep.EXPERIMENTS.values()))
check("a plain-baseline job gets a distinct id (ends with fs1|nolabel)", sweep.job_id(dict(sweep.PLAIN, seed=1)).endswith("|fs1|nolabel"))
try:
    sweep.experiment_jobs("phase99"); ok = False
except ValueError:
    ok = True
check("an unknown phase name raises ValueError", ok)

if os.path.exists("results.jsonl"):
    saved = [json.loads(line) for line in open("results.jsonl") if line.strip()]
    keys = ("mode", "window_len", "overlap", "w", "seed", "episodes", "first_stage", "label")
    check(f"job_id is stable on all {len(saved)} saved records (the record's own fields give back the same id)",
          all(sweep.job_id(r) == sweep.job_id({k: r[k] for k in keys if k in r}) for r in saved))
    in_phases = {sweep.job_id(j) for make in sweep.EXPERIMENTS.values() for j in make()}
    saved_ids = {sweep.job_id(r) for r in saved}
    print(f"info: {len(saved_ids & in_phases)} of {len(saved_ids)} saved results belong to phases 1-5 "
          "(runs added later by the notebook sweep are not in the phases)")
else:
    print("results.jsonl not found: the saved-results check is skipped")


=== G1. experiment lists and job ids ===
jobs per phase: {'phase1': 43, 'phase2': 32, 'phase3': 90, 'phase4': 60, 'phase5': 32}
  PASS  phase sizes are 43 / 32 / 90 / 60 / 32
  PASS  job ids are unique within each phase
  PASS  a plain-baseline job gets a distinct id (ends with fs1|nolabel)
  PASS  an unknown phase name raises ValueError
results.jsonl not found: the saved-results check is skipped


In [24]:
section("G2. train() and run_job() are reproducible and return what the analysis expects")
tiny = {**sweep.SETTINGS, "episodes": 40, "test_rollouts": 5, "R1": 3.0, "R2": 3.0}
job = dict(mode="robustness", window_len=1, overlap=0, w=0.5, seed=7)
tr = sweep.train(job, SYN_COURSE, tiny)
print(f"train(): {len(tr.ep_steps)} episodes, {len(tr.Q)} Q-states, {len(tr.visited)} visited states, {tr.train_s:.2f}s, steps/episode (mean) = {tr.ep_steps.mean():.0f}")
check("train() returns the monitor, the Q-table and per-episode arrays", len(tr.Q) > 0 and len(tr.ep_steps) == 40 and tr.monitor is not None)

a, b = sweep.run_job(job, SYN_COURSE, tiny), sweep.run_job(job, SYN_COURSE, tiny)
same_nan = lambda x, y: isinstance(x, float) and isinstance(y, float) and np.isnan(x) and np.isnan(y)
diff = [k for k in a if k != "train_s" and not (a[k] == b[k] or same_nan(a[k], b[k]))]
check("the same job run twice gives the identical record (except wall time)", not diff, f"differs in {diff}" if diff else f"{len(a) - 1} fields compared")
derived = {"auc_1500", "lap@500", "lap@1000", "lap@1500"}      # computed by load_results from the learning curve
missing = sorted((set(sweep.METRICS) - derived) - set(a))
check("the record contains every metric the analysis summarizes", not missing, f"missing {missing}" if missing else f"{len(sweep.METRICS) - len(derived)} metrics")
check("label=False runs and keeps the same reward settings", sweep.run_job(dict(job, label=False), SYN_COURSE, tiny)["hover_gain_per_step"] == a["hover_gain_per_step"])


=== G2. train() and run_job() are reproducible and return what the analysis expects ===
train(): 40 episodes, 738 Q-states, 738 visited states, 0.38s, steps/episode (mean) = 99
  PASS  train() returns the monitor, the Q-table and per-episode arrays
  PASS  the same job run twice gives the identical record (except wall time)   [29 fields compared]
  PASS  the record contains every metric the analysis summarizes   [18 metrics]
  PASS  label=False runs and keeps the same reward settings


True

In [25]:
section("G3. run_jobs resumes, and also works with worker processes")
import multiprocessing as mp
jobs = [dict(job, seed=s, episodes=20) for s in (1, 2, 3, 4)]
path = os.path.join(tempfile.mkdtemp(), "sweep_test.jsonl")
sweep.run_jobs(jobs, path, workers=1, log=lambda m: None, course_path=SYN_COURSE, settings=tiny)
check("4 jobs -> 4 result lines", sum(1 for _ in open(path)) == 4)
log = []
sweep.run_jobs(jobs, path, workers=1, log=log.append, course_path=SYN_COURSE, settings=tiny)
print("rerun said:", log[0])
check("rerunning skips finished jobs and adds no lines", " 0 to run" in log[0] and sum(1 for _ in open(path)) == 4)

path2 = os.path.join(tempfile.mkdtemp(), "sweep_pool_test.jsonl")
try:
    sweep.run_jobs(jobs, path2, workers=2, log=lambda m: None, course_path=SYN_COURSE, settings=tiny)
    ids = {sweep.job_id(json.loads(l)) for l in open(path2) if l.strip()}
    ok, detail = ids == {sweep.job_id(j, 20) for j in jobs}, f"start method: {mp.get_start_method()}"
except Exception as e:                     # report as a failed check instead of aborting the notebook
    ok, detail = False, f"{type(e).__name__}: {str(e)[:120]}"
check("2 worker processes run the same 4 jobs", ok, detail)


=== G3. run_jobs resumes, and also works with worker processes ===
  PASS  4 jobs -> 4 result lines
rerun said: 4 jobs, 4 results already in /var/folders/pr/twtdx5rn3f19t_p1dn_gkvz80000gn/T/tmpxd1tszt7/sweep_test.jsonl, 0 to run on 1 worker(s)
  PASS  rerunning skips finished jobs and adds no lines
  PASS  2 worker processes run the same 4 jobs   [start method: spawn]


True

In [26]:
section("G4. analysis helpers")
check("welch_t([1,2,3], [4,5,6]) = -3.674", abs(sweep.welch_t([1, 2, 3], [4, 5, 6]) + 3.6742) < 1e-3)
check("welch_t needs at least 2 values per group (returns NaN)", np.isnan(sweep.welch_t([1.0], [1, 2, 3])))
df = sweep.load_results(path)
summary = sweep.summarize(df)
print(summary[["cond", "mode", "window_len", "overlap", "w", "n_seeds", "lap_rate"]].to_string(index=False))
check("summarize() groups the 4 seeds into one STL condition", len(summary) == 1 and summary["n_seeds"].iloc[0] == 4 and summary["cond"].iloc[0] == "STL")
cmp_tbl = sweep.compare(df, "lap_steps", min_seeds=5)
check("compare() returns an empty table (with its columns) when no condition has enough seeds",
      cmp_tbl.empty and {"mode", "t_plain", "t_label"} <= set(cmp_tbl.columns))


=== G4. analysis helpers ===
  PASS  welch_t([1,2,3], [4,5,6]) = -3.674
  PASS  welch_t needs at least 2 values per group (returns NaN)
cond       mode  window_len  overlap   w  n_seeds  lap_rate
 STL robustness           1        0 0.5        4       0.0
  PASS  summarize() groups the 4 seeds into one STL condition
  PASS  compare() returns an empty table (with its columns) when no condition has enough seeds


True

## Summary

In [27]:
summary = (pd.DataFrame(CHECKS, columns=["section", "check", "passed"])
           .groupby("section", sort=False)["passed"].agg(checks="count", passed="sum").reset_index())
summary["failed"] = summary["checks"] - summary["passed"]
print(summary.to_string(index=False))
n_pass, n_all = sum(c[2] for c in CHECKS), len(CHECKS)
print(f"\n{n_pass}/{n_all} checks passed")
failed = [(s, n) for s, n, ok in CHECKS if not ok]
for s, n in failed:
    print("FAILED:", s, "->", n)
assert not failed, f"{len(failed)} check(s) failed"
print("ALL CHECKS PASSED")

                                                                                            section  checks  passed  failed
A1. until_robustness equals the definition sup_t' min(rho_target(t'), inf_{t''<=t'} rho_avoid(t''))       4       4       0
                                                                    A2. bad inputs raise ValueError       5       5       0
                  A3. window schedule: which states does each window cover, and when does it close?       1       1       0
                                                     A4. BHNRWindow lifecycle (start / push / stop)       5       5       0
                                        A5. H_t labels (class 0 is reserved for 'no window closed')       4       4       0
                                                                                  B. scenario signs       9       9       0
                                                                           C. masking of a bad step       4       4       0
        